- 3a Stochastic pulse distribution
- 3b Noise floor
- 3c Trigger and acquisition window
- 3d Charge integration
- 3e Neutron/gamma channels (3D plot)

In [ ]:
# Imports
# import pickle
from typing import Callable, Literal
from random import sample
from statistics import mean
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import mpl_toolkits.mplot3d.art3d as art3d
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
from scipy.stats import linregress
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn, get_df_col
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve

### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
# Functions
pileup_flag = 0x8000


def is_pileup_flag(flags: int) -> bool:
    return flags & pileup_flag != 0

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 30,
    max_adc: int = 16367,
    baseline_offset: float = 0.10
) -> pd.DataFrame:
    # offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()
    # baselines = signals_np[:, :baseline_idx_range].mean(axis=1).reshape(-1, 1)
    # signals_np = -signals_np + baselines + offset
    signals_np = -signals_np + max_adc
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def textbox(text, x, y, width, height, facecolor, textcolor, ax, text_x_offset=0.5, text_y_offset=0.5, ha="center"):
    rect_params = {
        "linewidth": 0,
        # "ec": "black",
        "fc": facecolor
    }
    text_params = {
        "ha": ha,
        "va": "center",
        "fontsize": fontsize,
        "color": textcolor
    }
    rect = mpl.patches.Rectangle((x, y), width, height, **rect_params)
    ax.add_patch(rect)
    # poly = mpl.patches.Polygon(
    ax.annotate(
        # (t_2 + t_3) / 2,
        text,
        (text_x_offset, text_y_offset),
        xycoords=rect,
        **text_params
    )

In [ ]:
# def get_gamma_neutron_color(
#     is_gamma: bool,
#     is_neutron: bool,
#     is_uncertain: bool,
#     psd: float,
#     gamma_color: ColorAlphaTuple,
#     neutron_color: ColorAlphaTuple
# ) -> ColorAlphaTuple | None:
#     if is_uncertain:
#         psd_unit_interval = to_unit_interval(psd, 0.2, 0.3)
#         return calculate_gradient_color(psd_unit_interval, gamma_color, neutron_color)
#     elif is_gamma:
#         return gamma_color
#     elif is_neutron:
#         return neutron_color
#     else:
#         return None


# def text3d(ax, xyz, s, zdir="z", size=None, angle=0, usetex=False, **kwargs):
#     """
#     Plots the string *s* on the Axes *ax*, with position *xyz*, size *size*,
#     and rotation angle *angle*. *zdir* gives the axis which is to be treated as
#     the third dimension. *usetex* is a boolean indicating whether the string
#     should be run through a LaTeX subprocess or not.  Any additional keyword
#     arguments are forwarded to `.transform_path`.

#     Originally created by matplotlib development team.
#     See https://matplotlib.org/stable/gallery/mplot3d/pathpatch3d.html

#     Note: zdir affects the interpretation of xyz.
#     """
#     x, y, z = xyz
#     if zdir == "y":
#         xy1, z1 = (x, z), y
#     elif zdir == "x":
#         xy1, z1 = (y, z), x
#     else:
#         xy1, z1 = (x, y), z

#     text_path = mpl.text.TextPath((0, 0), s, size=size, usetex=usetex)
#     trans = mpl.transforms.Affine2D().rotate(angle).translate(xy1[0], xy1[1])

#     p1 = mpl.patches.PathPatch(trans.transform_path(text_path), **kwargs)
#     ax.add_patch(p1)
#     art3d.pathpatch_2d_to_3d(p1, z=z1, zdir=zdir)


def get_bbox_center(bbox: mpl.transforms.BboxBase) -> tuple[float, float]:
    return (bbox.xmin + bbox.width / 2, bbox.ymin + bbox.height / 2)


def get_translation_to(
    pos_from: tuple[float, float],
    pos_to: tuple[float, float]
) -> tuple[float, float]:
    x_from, y_from = pos_from
    x_to, y_to = pos_to
    return (x_to - x_from, y_to - y_from)


def make_text_path(
    text: str | list[str],
    # position: tuple[float, float],
    # rotation: float,
    # scaling: tuple[float, float],
    linespacing: float,
    font_properties=None,
    usetex=False
) -> mpl.text.TextPath:
    # scaling_x, scaling_y = scaling
    # pos_x, pos_y = position
    paths = []
    
    if isinstance(text, str):
        text = [text]
    
    for i, line in enumerate(text):
        text_path = mpl.text.TextPath(
            (0, 0), line,
            prop=font_properties,
            size=1,
            usetex=usetex
        )
        bbox = text_path.get_extents()
        line_width = bbox.width
        from_x = bbox.xmin
        from_y = bbox.xmax
        # center first line on x=0, anchor at y=0, next lines below
        to_x = -line_width / 2
        to_y = -i * linespacing
        x = to_x - from_x
        y = to_y - from_y
        
        line_transform = mpl.transforms.Affine2D().translate(x, y)
        transformed_path = line_transform.transform_path(text_path)
        paths.append(transformed_path)
    
    text_path = mpl.path.Path.make_compound_path(*paths)
    return text_path


def make_bounding_box_for_text_path(
    text_path: mpl.text.TextPath,
    padding: float,
    rounding_size: float,
    zorder: int,
    mutation_aspect: float = 1,
    **bbox_params
) -> mpl.patches.FancyBboxPatch:
    text_bbox = text_path.get_extents()
    # get anchor corner, width, height
    # make FancyBboxPatch
    anchor = text_bbox.min
    boxstyle = f"round, pad={padding}, rounding_size={rounding_size}"
    return mpl.patches.FancyBboxPatch(
        anchor,
        text_bbox.width,
        text_bbox.height,
        boxstyle=boxstyle,
        mutation_aspect=mutation_aspect,
        **bbox_params
    )


def convert_text_path_to_patch(
    text_path: mpl.text.TextPath,
    zorder: int
) -> mpl.patches.PathPatch:
    return mpl.patches.PathPatch(text_path, ec="none", fc="k", zorder=zorder)


def patch_to_3d_plot_wall(
    patch: mpl.patches.Patch,
    ax: mpl.axes.Axes,
    zdir: str,
    z: float = 0,
):
    ax.add_patch(patch)
    art3d.pathpatch_2d_to_3d(patch, z=z, zdir=zdir)


def get_center_match_transform(
    box_from: mpl.transforms.Bbox,
    box_to: mpl.transforms.Bbox
) -> tuple[float, float]:
    from_c_x = (box_from.x0 + box_from.x1) / 2
    from_c_y = (box_from.y0 + box_from.y1) / 2
    to_c_x = (box_to.x0 + box_to.x1) / 2
    to_c_y = (box_to.y0 + box_to.y1) / 2
    return (to_c_x - from_c_x, to_c_y - from_c_y)

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data.get(ExperimentDataKey.UNCLASSIFIED)
    if unclassified_df is not None:
        unclassified_df = load.calculate_timetag_hours(unclassified_df)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data.get(ExperimentDataKey.UNCLASSIFIED)
    if unclassified_df is not None:
        unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

### Figure 3c/d Processing

#### Neutron Classification

In [ ]:
exp_id = "TB-26"

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
Z, xe, ye = proc.get_psd_energy_histogram(
    psd_report,
    calibrated_energy_column,
    energy_width=energy_width
)
exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# TODO get fit dataframe (not needed if loading, but do anyway to keep process consistent)
exp_data = experiment_neutron_data[exp_id]
stop_here = False

exp_data = experiment_neutron_data[exp_id]
Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

# # Default
# default_bounds: BimodalBounds = (
#     BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
#     BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
# )

# bounds_a: BimodalBounds = (
#     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
#     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
# )

# bounds_b: BimodalBounds = (
#     BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
#     BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
# )

# # Ranged Example
# bounds = [
#     ((0, 60), bounds_a),
# ]

df, df_err = proc.scan_histogram_slices(
    Z,
    xe,
    ye,
    fit_style="peak_finder",
    # default_bounds,
    # bounds=bounds,
    start_idx=start_scan_idx,
    end_idx=end_scan_idx
)
df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

if bad_slice_indexes is not None:
    exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
    exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
    stop_here = True
else:
    # exp_data['fom_results'] = df
    exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
exp_data = experiment_neutron_data[exp_id]
if ExperimentDataKey.FOM_RESULTS not in exp_data:
    print(f"No good fit data on Experiment {exp_id}")
else:
    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]
    
    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()
    
    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
borders = exp_data[ExperimentDataKey.BORDERS]

psd_report = proc.classify(
    psd_report,
    calibrated_energy_column,
    borders,
    DetectorDataframeColumn.NEW_N_CLASS
)

exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

#### Pulse Selection

In [ ]:
exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value

gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
neutrons_only = psd_report.query(n_class_col_name).copy()
exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
exp_data = experiment_neutron_data[exp_id]
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
signals_df = exp_data["signals_df"]

neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
gamma_signals = signals_df.loc[gamma_only.index].astype("int32")
print(neutron_signals.shape)
print(neutron_signals.iloc[0, :])

# n_signals_np = neutron_signals.to_numpy()
# n_baselines = n_signals_np.max(axis=1).reshape(-1, 1)
# n_signals_np = -n_signals_np + n_baselines
# print(n_signals_np.max())
# neutron_signals = pd.DataFrame(n_signals_np, index=neutron_signals.index, columns=neutron_signals.columns)
neutron_signals = correct_raw_signals(neutron_signals)
print(neutron_signals.iloc[0, :])

# g_signals_np = gamma_signals.to_numpy()
# g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
# g_signals_np = -g_signals_np + g_baselines
# print(g_signals_np.max())
# gamma_signals = pd.DataFrame(g_signals_np, index=gamma_signals.index, columns=gamma_signals.columns)
gamma_signals = correct_raw_signals(gamma_signals)

exp_data["neutron_signals"] = neutron_signals
exp_data["gamma_signals"] = gamma_signals

In [ ]:
min_height = 2500
max_height = 3500

exp_data = experiment_neutron_data[exp_id]
signals = exp_data["neutron_signals"]
# signals = exp_data["gamma_signals"]

selected = None

bad_signals = []

for signal_id, signal in signals.iterrows():
    # get clean neutron pulse (no secondary peak)
    if selected is not None:
        break
    if signal_id in bad_signals:
        continue
    sig_height = signal.max()
    if sig_height < min_height or sig_height > max_height:
        continue
    else:
        print(f"Signal ID = {signal_id}, height = {sig_height}")
        selected = signal_id, signal

if selected is None:
    raise Exception("No pulse found")
exp_data["selected_signal"] = selected

In [ ]:
noise_floor_offset = 100
_, selected_signal = exp_data["selected_signal"]
smoothed_signal = savgol_filter(selected_signal, 11, 5)
noise_floor_bottom = smoothed_signal - noise_floor_offset
noise_floor_top = smoothed_signal + noise_floor_offset
exp_data["noise_floor_band"] = (noise_floor_bottom, noise_floor_top)

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
exp_data = experiment_neutron_data["TB-26"]
selected_id, selected_signal = exp_data["selected_signal"]
noise_floor_bottom, noise_floor_top = exp_data["noise_floor_band"]

signal_y = selected_signal.values
signal_x = np.arange(0, len(signal_y)) * 2

fig, ax = plt.subplots(figsize=(12, 8))
ax.plot(signal_x, signal_y, lw=3)
ax.plot(signal_x, noise_floor_bottom, lw=3)
ax.plot(signal_x, noise_floor_top, lw=3)
# ax.set_xlim(75, 125)
# ax.set_ylim(2800, 3100)